In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import math

def mostrar_resultados_test(ruta_carpeta, nombre_archivo, titulo):
    if not nombre_archivo:
        print(f"\n No se encontró archivo para: {titulo}")
        return

    ruta = os.path.join(ruta_carpeta, nombre_archivo)
    
    try:
        df = pd.read_csv(ruta)
        
        # Nos quedamos con la última fila (resultados finales)
        datos = df.iloc[-1]
        
        print("\n" + "-"*60)
        print(f"  {titulo.upper()}")
        print(f"    Archivo: {nombre_archivo}")
        print("-"*60)

        # Definimos las categorías basándonos en los prefijos de las columnas
        categorias = {
            "GLOBAL METRICS": "global_",
            "PRIMARY TASK": "primary_",
            "SECONDARY TASK": "secondary_",
            "MULTILABEL METRICS": "multilabel_"
        }

        # Iteramos por cada categoría para imprimir ordenadamente
        for nombre_cat, prefijo in categorias.items():
            # Filtramos las métricas que empiezan por el prefijo actual
            metricas_grupo = [k for k in datos.index if k.startswith(prefijo)]
            
            if metricas_grupo:
                print(f"\n{nombre_cat}")
                print(f"{'Metrica':<35} | {'Valor':<10}")
                print("-" * 50)
                
                for key in metricas_grupo:
                    valor = datos[key]
                    # Limpiamos el nombre: quitamos el prefijo y ponemos mayúsculas bonitas
                    nombre_limpio = key.replace(prefijo, "").replace("_", " ").title()
                    
                    # Formateo condicional: si es float lo mostramos con 4 decimales
                    if isinstance(valor, (float, int)):
                        val_str = f"{valor:.4f}"
                    else:
                        val_str = str(valor)
                        
                    print(f"  • {nombre_limpio:<31} : {val_str}")

        print("\n" + "═"*60 + "\n")

    except Exception as e:
        print(f"Error leyendo {nombre_archivo}: {e}")

def analizar_entrenamiento_y_test(ruta_carpeta):
    if not os.path.exists(ruta_carpeta):
        print(f"La ruta '{ruta_carpeta}' no existe.")
        return

    archivos = [f for f in os.listdir(ruta_carpeta) if f.endswith('.csv')]
    
    # 1. Identificar los archivos por nombre
    f_train = next((f for f in archivos if "train_train" in f), None)
    f_val   = next((f for f in archivos if "train_validation" in f), None)
    f_test  = next((f for f in archivos if "train_test" in f), None)
    f_sicap = next((f for f in archivos if "sicap_test" in f), None)

    # --- PARTE 1: GRÁFICAS EVOLUTIVAS (Train vs Validation) ---
    if f_train and f_val:
        print(f"Generando gráficas comparativas de:")
        print(f"   • Train: {f_train}")
        print(f"   • Val:   {f_val}\n")
        
        df_t = pd.read_csv(os.path.join(ruta_carpeta, f_train)).sort_values('epoch')
        df_v = pd.read_csv(os.path.join(ruta_carpeta, f_val)).sort_values('epoch')
        
        # Definir grupos de métricas para organizar las gráficas
        grupos_metricas = {
            "Multilabel Metrics": ['multilabel_subset_accuracy', 'multilabel_precision_micro', 'multilabel_recall_micro', 
                                   'multilabel_f1_micro', 'multilabel_precision_macro', 'multilabel_recall_macro', 
                                   'multilabel_f1_macro', 'multilabel_auroc_macro', 'multilabel_auprc_macro'],
            "Global Metrics": ['global_accuracy', 'global_precision', 'global_recall', 'global_f1', 'global_kappa'],
            "Primary Task": ['primary_accuracy', 'primary_precision', 'primary_recall', 'primary_f1', 'primary_kappa'],
            "Secondary Task": ['secondary_accuracy', 'secondary_precision', 'secondary_recall', 'secondary_f1', 'secondary_kappa'],
            
        }

        for nombre_grupo, columnas in grupos_metricas.items():
            # Filtrar columnas que realmente existen en el CSV
            cols_existentes = [c for c in columnas if c in df_t.columns and c in df_v.columns]
            
            if not cols_existentes:
                continue

            # Configuración de subplots (calculamos filas necesarias)
            n_cols = 3
            n_rows = math.ceil(len(cols_existentes) / n_cols)
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
            fig.suptitle(f'Evolución: {nombre_grupo} (Train vs Val)', fontsize=16)
            axes = axes.flatten()

            for i, col in enumerate(cols_existentes):
                ax = axes[i]
                ax.plot(df_t['epoch'], df_t[col], label='Train', marker='.', linestyle='-')
                ax.plot(df_v['epoch'], df_v[col], label='Validation', marker='.', linestyle='--')
                
                ax.set_title(col)
                ax.set_xlabel('Epoch')
                ax.set_ylabel('Valor')
                ax.legend()
                ax.grid(True, alpha=0.3)

            # Ocultar subplots vacíos si los hay
            for j in range(i + 1, len(axes)):
                axes[j].axis('off')

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
    else:
        print("No se encontraron los archivos 'train_train' y 'train_validation' para graficar.")

    # --- PARTE 2: RESULTADOS FINALES (Test y Sicap) ---
    print("\n" + "="*80)
    print("RESULTADOS FINALES EN DATASETS DE PRUEBA")
    print("="*80)

    mostrar_resultados_test(ruta_carpeta, f_test, "TEST INTERNO (Train Test)")
    mostrar_resultados_test(ruta_carpeta, f_sicap, "TEST EXTERNO (SICAP)")

# --- EJECUCIÓN ---


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

import numpy as np
from PIL import Image
import os
import pandas as pd
import random
import sys 
from tqdm import tqdm 
import gc 
import h5py

from accelerate import Accelerator
from torch.cuda.amp import autocast
from safetensors.torch import load_file



# Directorio actual
current_dir  = os.path.abspath(os.getcwd())
sys.path.append(os.path.dirname(current_dir))

from utils.utils_train import  loss_restrictions, select_gleason_grades_from_batch_proportions, selecting_abs_threshold, selecting_prob_thresholds
from utils.utils_metrics import compute_all_gleason_metrics
from models.keep_model import get_keep_ia3_model, device

data_sicap_patches='/workspace/data/features_SICAP/patches'
data_sicap_path='/workspace/data/wsi_labels_changed_SICAP.xlsx'
 
accelerator = Accelerator()
device = accelerator.device
print(f"Usando dispositivo: {device}", flush = True)

model = get_keep_ia3_model(clasification_model = 'mlp',pretrain=False )
ckpt_path = "/workspace/results/try_0.0001_16_100_opt_threshold/best_model/model.safetensors"
state_dict = load_file(ckpt_path)
model.load_state_dict(state_dict, strict=True)

print("✓ Pesos cargados correctamente")
#cargar este modleo estos pssos
loss = nn.BCELoss() #WithLogitsLoss(pos_weight=sum_weights) 
def create_file_label_df(images_path, data_path, NAME='', small_dataset=False, slide_col='image_id', num_samples=10):
    patch_list, patient_list = [], []
    gleason_score = []
    gleason_primary_list, gleason_secondary_list = [], []
    
    df = pd.read_excel(data_path)
    df = df.head(10) if small_dataset else df

    wsis = [f for f in os.listdir(images_path) if f.endswith('.h5')]

    # ---- 🔥 Seleccionar solo 10 WSIs ----
    wsis = wsis[:num_samples]      # usa los primeros
    # wsis = random.sample(wsis, num_samples)   # o aleatorios

    for f in tqdm(wsis, desc="Procesando WSIs", file=sys.stdout):
        slide_id = f.split('.h5')[0]
        if NAME == 'SICAP':
            slide_id = slide_id.split('_')[0]

        row = df[df[slide_col] == slide_id]
        if not row.empty:
            ruta_h5 = os.path.join(images_path, f)

            gleason = row[['gleason_score']].values.item()
            patient_id = row['patient_id'].values[0] if 'patient_id' in df.columns else row['image_id'].values[0]

            patch_list.append(ruta_h5)
            patient_list.append(patient_id)
            gleason_score.append(gleason)
            gleason_primary_list.append(row['gleason_primary'].values[0])
            gleason_secondary_list.append(row['gleason_secondary'].values[0])

    df_train = pd.DataFrame({
        'patch': patch_list,
        'patients': patient_list,
        'gleason_score': gleason_score,
        'gleason_primary': gleason_primary_list,
        'gleason_secondary': gleason_secondary_list
    })

    return df_train

class RAM_WSI_Dataset(Dataset):
    def __init__(self, dataframe, indices, random_subsets=False, len_subset=1000):
        gc.collect()
        self.dataframe = dataframe
        self.indices = list(indices)
        self.random_subsets = bool(random_subsets)

        if self.random_subsets:
            self.len_subset = min(int(len_subset), len(self.indices))
        else:
            self.len_subset = len(self.indices)

        self.transform = T.Compose([
            T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(size=(224, 224)),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])

        self.images, self.labels, self.coords = [], [], []

        for i in tqdm(self.indices, desc="Loading WSI", file=sys.stdout):
            row = self.dataframe.iloc[i]

            label = torch.tensor(
                [row['gleason_primary_mapped'], row['gleason_secondary_mapped']],
                dtype=torch.long
            )
            self.labels.append(label)

            with h5py.File(row['patch'], "r", swmr=True) as h5_file:
                imgs = h5_file["imgs"][:]        # (N, H, W, C)
                coords = h5_file["coords"][:]    # (N, 2)

                if imgs.shape[0] <= 1:
                    continue

            self.images.append(imgs)
            self.coords.append(coords)

    def __len__(self):
        return self.len_subset

    def __getitem__(self, idx):
        imgs = self.images[idx]
        coords = self.coords[idx]
        label = self.labels[idx]

        # Transformar cada patch
        imgs = torch.stack([self.transform(Image.fromarray(im)) for im in imgs])

        return imgs, coords, label
def collate_fn(batch):
    features,coords, labels = zip(*batch)

    list_features = list(features) # features: tuple de tensores de distinta longitud -> devolvemos como lista
    list_coords = list(coords)
    labels = torch.stack(labels, dim=0) # Convertir labels a tensor
    return list_features, list_coords, labels
def run_epoch(model, dataloader, device, accelerator, 
              loss_BCE, optimizer=None, 
              args=None, mode="train"):

    assert mode in ["train", "val", "test"], "mode debe ser 'train', 'val' o 'test'."

    # Acumuladores métricas
    all_y_true_primary, all_y_true_secondary = [], []
    all_y_pred_primary, all_y_pred_secondary = [], []
    all_probs, all_y_true_onehot = [], []
    all_wsi_proportions = []

    # Para TEST → guardar coords + scores + patches
    test_outputs = []   # ← NUEVO

    total_loss_sum = 0.0
    total_seen = torch.tensor(0, device=device)

    model.train() if mode == "train" else model.eval()
    torch.set_grad_enabled(mode == "train")

    for (patch_imgs, coords, y) in tqdm(
        dataloader, desc=f"{mode.upper()}",
        disable=not accelerator.is_local_main_process,
        file=sys.stdout
    ):
        """
        patch_imgs:   (B, N_patches, 3, 224, 224)
        coords:       list de B arrays  [N_patches, 2]
        y:            (B,2)
        slide_ids:    lista de strings
        """

        y_primary = y[:, 0].to(device)
        y_secondary = y[:, 1].to(device)
        y = y.to(device)

        # One-hot
        y_onehot = torch.zeros(y.shape[0], 4, device=device)
        y_onehot.scatter_(1, y, 1)

        with torch.no_grad() if mode == "test" else autocast(dtype=torch.float16):

            # --- Forward ---
            wsi_prediction, patch_predictions = model(patch_imgs)
            # wsi_prediction: (B,4)
            # patch_predictions: list of B tensors → (N_i,4)

        # Proporciones
        wsi_proportions = torch.stack([
            p.mean(dim=0) if p.numel() > 0 else torch.zeros(
                wsi_prediction.size(1), device=wsi_prediction.device)
            for p in patch_predictions
        ], dim=0)

        # Pérdida
        loss = loss_BCE(wsi_prediction, y_onehot)
        loss_pe_pc = loss_restrictions(wsi_proportions, args.alpha_pe, args.alpha_pc)
        total_loss = loss + loss_pe_pc

        if mode == "train":
            optimizer.zero_grad()
            accelerator.backward(total_loss)
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_sz = len(patch_imgs)
        total_seen += batch_sz
        total_loss_sum += accelerator.gather(total_loss.detach() * batch_sz).sum().item()

        # Predicción primaria/secundaria
        pred_primary, pred_secondary = select_gleason_grades_from_batch_proportions(
            wsi_proportions, args.tumor_abs_thresh)

        # Guardar para métricas
        all_y_true_primary.append(y_primary.cpu())
        all_y_true_secondary.append(y_secondary.cpu())
        all_y_pred_primary.append(pred_primary.cpu())
        all_y_pred_secondary.append(pred_secondary.cpu())
        all_y_true_onehot.append(y_onehot.cpu())
        all_probs.append(wsi_prediction.detach().cpu())

        # 🔥🔥🔥 NUEVO: si estamos en TEST → guardar datos para heatmap
        if mode == "test":
            for i in range(batch_sz):

                slide_out = {
                    "slide_id": patch_imgs[i].detach().cpu().numpy(),
                    "coords": coords[i].numpy(),
                    # score = prob. del tumor global → si quieres otra cosa lo cambiamos
                    "scores": patch_predictions[i][:, 1].detach().cpu().numpy(),
                    
                }
                test_outputs.append(slide_out)

    # ---- MÉTRICAS ----
    total_seen = accelerator.reduce(total_seen, reduction="sum").item()
    epoch_loss = total_loss_sum / total_seen

    y_true1 = torch.cat(all_y_true_primary).numpy()
    y_true2 = torch.cat(all_y_true_secondary).numpy()
    y_pred1 = torch.cat(all_y_pred_primary).numpy()
    y_pred2 = torch.cat(all_y_pred_secondary).numpy()
    probs = torch.cat(all_probs).numpy()
    y_onehot = torch.cat(all_y_true_onehot).numpy()

    metrics = compute_all_gleason_metrics(
        y_true_primary=y_true1,
        y_pred_primary=y_pred1,
        y_true_secondary=y_true2,
        y_pred_secondary=y_pred2,
        y_true_onehot=y_onehot,
        y_probs=probs,
        y_prob_threshold=args.y_prob_thresh
    )

    if mode == "val":
        val_metrics, y_prob_threshold = selecting_prob_thresholds(
            y_true1, y_pred1, y_true2, y_pred2, y_onehot, probs
        )
        metrics_global, tumor_abs_threshold = selecting_abs_threshold(
            all_wsi_proportions, y_true1, y_true2
        )
        return metrics, epoch_loss, y_prob_threshold, tumor_abs_threshold

    if mode == "test":
        # 🔥 En test devolvemos test_outputs para hacer heatmaps
        return metrics, epoch_loss, test_outputs

    return metrics, epoch_loss

def build_heatmap(coords, scores, patch_size=512, scale_factor=16, save_path=None):
    """
    coords: Nx2 array (x, y)
    scores: N array (peso o score por patch)
    save_path: path para guardar el heatmap
    """

    # Escalar coordenadas
    coords = coords // scale_factor
    max_x, max_y = coords[:, 0].max(), coords[:, 1].max()
    min_x, min_y = coords[:, 0].min(), coords[:, 1].min()

    width, height = max_x - min_x + 1, max_y - min_y + 1

    shifted_coords = coords.copy()
    shifted_coords[:, 0] -= min_x
    shifted_coords[:, 1] -= min_y

    # Inicializar heatmap
    heatmap = np.zeros((height + patch_size, width + patch_size), dtype=float)

    # Añadir valores por patch
    for (x, y), w in zip(shifted_coords, scores):
        heatmap[y:y+patch_size, x:x+patch_size] += w

    # Normalizar
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    heatmap = (255 * heatmap).astype(np.uint8)
    img = Image.fromarray(heatmap, mode="L")

    if save_path:
        img.save(save_path)

    return img

df_test = create_file_label_df(data_sicap_patches, data_sicap_path, NAME = 'SICAP')

label_mapping = {0: 0, 1: 0, 2: 0, 3: 1, 4: 2, 5: 3}
df_test['gleason_primary_mapped'] = df_test['gleason_primary'].map(label_mapping)
df_test['gleason_secondary_mapped'] = df_test['gleason_secondary'].map(label_mapping)
df_test['gleason_score'] = df_test['gleason_score'].astype(int)

print("Loading embeddings for Inference ...")
inference_loader = DataLoader(RAM_WSI_Dataset(df_test, df_test.index), shuffle=False, batch_size=4, collate_fn=collate_fn)

inference_metrics, inference_loss, metrics, epoch_loss, test_outputs= run_epoch(model, inference_loader, device, accelerator, loss, None, mode='test')   

print(f" Test Loss: {inference_loss:.4f}  Acc_global: {inference_metrics['global'][0]:.4f} Kappa_global: {inference_metrics['global'][-1]:.4f} | Acc_primary: {inference_metrics['primary'][0]:.4f} Kappa_primary: {inference_metrics['primary'][-1]:.4f} |Acc_secondary: {inference_metrics['secondary'][0]:.4f} Kappa_secondary: {inference_metrics['secondary'][-1]:.4f}\n")
print(f"Acuracy: {inference_metrics[0]}, | Precision: {inference_metrics[1]} | Recall : {inference_metrics[2]}| F1 Score: {inference_metrics[3]}")
print(f"Métricas : {inference_metrics}")

img = build_heatmap(test_outputs['coords'], test_outputs['scores'], patch_size=512, scale_factor=16, save_path=None)

img.show()

Usando dispositivo: cuda


trainable params: 0 || all params: 414,259,969 || trainable%: 0.0
Procesando WSIs: 100%|██████████| 10/10 [00:00<00:00, 596.75it/s]
Loading embeddings for Inference ...
TEST:   0%|          | 0/3 [00:00<?, ?it/s]


AttributeError: 'list' object has no attribute 'to'